# 3. Machine Learning: Faire Miete & Deal-Bewertung
Ziel: Für jede Wohnung schätzen wir die "faire" Miete aus Fläche, Zimmern, Bezirk, Freifläche, Befristung und Öffi-Score (Plattform als Kontrollvariable). Durch den Vergleich mit der tatsächlichen Miete erkennen wir gute Deals und zu teure Wohnungen.
Beantwortet werden die Fragen 2 (Vorhersagequalität), 3 (Einflussfaktoren), 4 und 5 (Deal-Kategorien) sowie 6 (Stabilität über die Zeit).

##  Daten laden
Wir starten eine Spark-Session und laden dabei den MongoDB Spark Connector (`spark.jars.packages`), damit Spark direkt aus MongoDB lesen kann. Beim ersten Start wird der Connector automatisch heruntergeladen.
Spark liest die Collection `listings_clean` aus der Datenbank `wohnungsmarkt`. Anschließend wandeln wir die Daten mit `toPandas()` in eine pandas-Tabelle um, da scikit-learn mit pandas arbeitet.

In [1]:
import pandas as pd
import numpy as np
from pymongo import MongoClient

def check(bedingung, text):
    print("✅" if bedingung else "❌", text)

coll = MongoClient("mongodb://localhost:27017/")["wohnungsmarkt"]["listings_clean"]
df = pd.DataFrame(list(coll.find({})))
check(len(df) > 0, f"{len(df)} Inserate geladen")

✅ 38 Inserate geladen


## 3.1 Daten laden
Wir lesen die bereinigte Collection `listings_clean` aus der MongoDB-Datenbank `wohnungsmarkt` über Spark ein und wandeln sie mit `toPandas()` in eine pandas-Tabelle um, da scikit-learn mit pandas arbeitet.
Die Hilfsfunktion `check` gibt bei Prüfungen ✅ oder ❌ aus.

## 3.2 Öffi-Score
Für jedes Inserat zählen wir die Linien in der Umgebung und gewichten sie nach Verkehrsmittel: U-Bahn 5, S-Bahn 4, Straßenbahn 3 und Bus 2 Punkte. Die Summe normieren wir auf eine Skala von 0 bis 10, wobei die bestangebundene Wohnung 10 bekommt.
Vereinfachung: Die Distanzgewichtung (≤ 300 m voll, ≤ 600 m halb) ist hier noch nicht enthalten.

In [2]:
punkte = (5 * df["n_ubahn_lines"] + 4 * df["n_sbahn_lines"]
          + 3 * df["n_tram_lines"] + 2 * df["n_bus_lines"])
grenze = punkte.quantile(0.95)
df["oeffi_score"] = (punkte / grenze * 10).clip(upper=10).round(1)

check(df["oeffi_score"].between(0, 10).all(), "Öffi-Score liegt zwischen 0 und 10")
df["oeffi_score"].describe()

✅ Öffi-Score liegt zwischen 0 und 10


count    38.000000
mean      2.815789
std       2.485039
min       0.100000
25%       1.425000
50%       1.950000
75%       3.875000
max      10.000000
Name: oeffi_score, dtype: float64

## 3.3 Daten vorbereiten
Wir behalten nur Inserate mit Preis, Fläche, Zimmern und Bezirk.

Sonderfälle entfernen: Einige Inserate sind keine regulären Mietwohnungen, z. B. WG-Zimmer (Miete für ein Zimmer, aber Fläche der ganzen Wohnung), Business-Apartments (Kurzzeitmiete), Häuser oder Mietkauf-Angebote. Würde das Modell mit ihnen trainieren, würde das die faire Miete für alle anderen Wohnungen verzerren. Wir erkennen sie per Schlagwortsuche (regulärer Ausdruck) im Titel. Der Filter wurde iterativ verfeinert, da Begriffe wie "Haus" oder "WG" auch in regulären Inseraten vorkommen (z. B. "WG-geeignet", "saniertes Haus").

Danach entfernen wir extreme Ausreißer (billigste und teuerste 1 %).
Als Zielvariable verwenden wir die logarithmierte Miete `log_price`. Mieten sind schief verteilt, weil wenige sehr teure Wohnungen dabei sind. Der Logarithmus macht die Verteilung gleichmäßiger, und die Koeffizienten lassen sich später als Prozent-Effekte lesen.

In [3]:
df = df.dropna(subset=["price", "area_m2", "rooms", "bezirk"])

# Sonderfälle entfernen
muster = r"\bWG\b(?![\s-]*(geeignet|taugli))|WG-Zimmer|Studenten|Semester|Business|Einfamilienhaus|Reihenhaus|^Haus\b|Mietkauf"
sonderfall = df["title"].str.contains(muster, case=False, na=False)
print("Entfernte Sonderfälle:", sonderfall.sum())
print("Beispiele:", list(df.loc[sonderfall, "title"].head(5)))
df = df[~sonderfall]

# Ausreißer entfernen
df = df[(df["price"] > df["price"].quantile(0.01)) & (df["price"] < df["price"].quantile(0.99))]
df["log_price"] = np.log(df["price"])

check(df["price"].min() > 0, "Alle Preise positiv")
check(len(df) > 1000, f"Genug Daten ({len(df)} Inserate)")

Entfernte Sonderfälle: 0
Beispiele: []
✅ Alle Preise positiv
❌ Genug Daten (36 Inserate)


C:\Users\clara\AppData\Local\Temp\ipykernel_38000\2822200567.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sonderfall = df["title"].str.contains(muster, case=False, na=False)


## 3.4 Features festlegen
X enthält die Merkmale, y die Zielvariable.
Bezirk und Plattform sind Text, deshalb wandeln wir sie mit `get_dummies` in 0/1-Spalten um (One-Hot-Encoding). Freifläche und privater Anbieter sind bereits 0/1-Werte.
Die Befristung wird nur verwendet, wenn die Spalte vorhanden ist.

In [4]:
features = ["area_m2", "rooms", "has_outdoor_space", "oeffi_score", "bezirk", "source"]
if "befristet" in df.columns:
    features.append("befristet")

basis = df["bezirk"].value_counts().idxmax()   # Bezirk mit den meisten Inseraten
print("Vergleichsbezirk:", basis)

X = pd.get_dummies(df[features], columns=["bezirk", "source"]).astype(float)
X = X.drop(columns=[f"bezirk_{basis}", "source_immoscout24"])
y = df["log_price"]

print("Features:", list(X.columns))
check(X.isna().sum().sum() == 0, "Keine fehlenden Werte in den Features")
check("befristet" in features, "Befristung ist als Feature enthalten")
check(len(X) == len(y), f"X und y gleich lang ({len(X)})")

Vergleichsbezirk: 3
Features: ['area_m2', 'rooms', 'has_outdoor_space', 'oeffi_score', 'bezirk_2', 'bezirk_4', 'bezirk_5', 'bezirk_9', 'bezirk_10', 'bezirk_12', 'bezirk_13', 'bezirk_14', 'bezirk_15', 'bezirk_16', 'bezirk_18', 'bezirk_19', 'bezirk_21', 'bezirk_22', 'bezirk_23', 'source_willhaben']
✅ Keine fehlenden Werte in den Features
❌ Befristung ist als Feature enthalten
✅ X und y gleich lang (36)


## 3.5 Train/Test-Split
80 % der Daten nutzen wir zum Trainieren, 20 % zum Testen mit Wohnungen, die das Modell nicht kennt. `random_state=42` macht die Aufteilung reproduzierbar.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
check(len(X_train) + len(X_test) == len(X), "Train + Test = alle Daten")

✅ Train + Test = alle Daten


## 3.6 Modelle trainieren und bewerten (Frage 2)
Wir vergleichen drei Modelle:
- Lineare Regression: einfache Baseline, gut interpretierbar
- Random Forest: viele Entscheidungsbäume, deren Ergebnisse gemittelt werden
- Gradient Boosting: Bäume werden nacheinander gebaut, jeder korrigiert die Fehler des vorherigen
Da die Modelle log-Mieten vorhersagen, rechnen wir mit `np.exp` zurück in Euro. Kennzahlen:
- MAE: durchschnittlicher Fehler in €
- MAPE: durchschnittlicher Fehler in %
- RMSE: wie MAE, bestraft große Fehler aber stärker
- R²: Anteil der erklärten Preisunterschiede (1 = perfekt)

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score

modelle = {
    "Lineare Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

ergebnisse = []
for name, modell in modelle.items():
    modell.fit(X_train, y_train)
    echt = np.exp(y_test)
    prognose = np.exp(modell.predict(X_test))
    ergebnisse.append({
        "Modell": name,
        "MAE (€)": mean_absolute_error(echt, prognose),
        "MAPE (%)": mean_absolute_percentage_error(echt, prognose) * 100,
        "RMSE (€)": mean_squared_error(echt, prognose) ** 0.5,
        "R²": r2_score(echt, prognose),
    })

ergebnisse = pd.DataFrame(ergebnisse).round(2)
ergebnisse

check(ergebnisse["R²"].max() > 0.5, f"Bestes R² über 0.5 ({ergebnisse['R²'].max():.2f})")
check(ergebnisse["MAPE (%)"].min() < 20, f"Fehler unter 20 % ({ergebnisse['MAPE (%)'].min():.1f} %)")

❌ Bestes R² über 0.5 (0.38)
✅ Fehler unter 20 % (18.1 %)


## 3.7 Cross-Validation
Ein einzelner Split kann Glück oder Pech sein. Bei der 5-fachen Cross-Validation werden die Daten in 5 Teile geteilt, jedes Teil ist einmal Testset. Ist der Mittelwert ähnlich wie oben und die Streuung klein, ist das Ergebnis stabil.

In [7]:
from sklearn.model_selection import cross_val_score

for name, modell in modelle.items():
    scores = cross_val_score(modell, X, y, cv=5, scoring="r2")
    print(f"{name}: R² = {scores.mean():.3f} (± {scores.std():.3f})")
    check(scores.std() < 0.05, f"{name} ist stabil über alle 5 Teile")

bester_name = ergebnisse.sort_values("MAE (€)").iloc[0]["Modell"]
bestes = modelle[bester_name]
print("Bestes Modell:", bester_name)

Lineare Regression: R² = 0.259 (± 0.381)
❌ Lineare Regression ist stabil über alle 5 Teile
Random Forest: R² = 0.674 (± 0.281)
❌ Random Forest ist stabil über alle 5 Teile
Gradient Boosting: R² = 0.671 (± 0.326)
❌ Gradient Boosting ist stabil über alle 5 Teile
Bestes Modell: Random Forest


## 3.8 Einflussfaktoren: Prozent-Effekte (Frage 3)
Weil die lineare Regression auf die log-Miete trainiert wurde, bedeutet ein Koeffizient b: Steigt das Merkmal um 1, ändert sich die Miete um (e^b − 1) · 100 %.
Beispiel: b = 0,08 bei Freifläche heißt, dass eine Freifläche die Miete um ca. 8 % erhöht, bei sonst gleichen Merkmalen.

In [8]:
lin = modelle["Lineare Regression"]
effekte = pd.Series((np.exp(lin.coef_) - 1) * 100, index=X.columns).round(1)
effekte[["area_m2", "rooms", "has_outdoor_space", "oeffi_score"]]
effekte.filter(like="bezirk_").sort_values(ascending=False)

bezirk_19    10.3
bezirk_18     4.7
bezirk_12     0.0
bezirk_14     0.0
bezirk_10    -3.4
bezirk_21   -11.7
bezirk_16   -16.8
bezirk_22   -16.9
bezirk_4    -21.0
bezirk_15   -21.9
bezirk_13   -22.4
bezirk_9    -31.0
bezirk_23   -32.7
bezirk_2    -34.8
bezirk_5    -36.4
dtype: float64

## 3.9 Permutation Importance
Für das beste Modell mischen wir die Werte eines Merkmals zufällig durch und messen, wie viel schlechter die Vorhersage wird. Je größer der Verlust, desto wichtiger ist das Merkmal.

In [9]:
from sklearn.metrics import r2_score
import plotly.express as px

gruppen = {
    "Fläche": ["area_m2"],
    "Zimmer": ["rooms"],
    "Freifläche": ["has_outdoor_space"],
    "Öffi-Score": ["oeffi_score"],
    "Bezirk": [s for s in X.columns if s.startswith("bezirk_")],
    "Plattform": [s for s in X.columns if s.startswith("source_")],
}

basis_r2 = r2_score(y_test, bestes.predict(X_test))
rng = np.random.default_rng(42)
wichtigkeit = {}
for name, spalten in gruppen.items():
    verluste = []
    for _ in range(10):
        X_gemischt = X_test.copy()
        zufall = rng.permutation(len(X_test))
        X_gemischt[spalten] = X_test[spalten].values[zufall]
        verluste.append(basis_r2 - r2_score(y_test, bestes.predict(X_gemischt)))
    wichtigkeit[name] = np.mean(verluste)

wichtigkeit = pd.Series(wichtigkeit).sort_values()
px.bar(x=wichtigkeit.values, y=wichtigkeit.index, orientation="h",
       labels={"x": "Verlust an R² beim Mischen", "y": ""},
       title=f"Wichtigste Merkmale ({bester_name})").show()

## 3.10 Vorhersage vs. tatsächliche Miete
Jeder Punkt ist eine Wohnung aus dem Testset. Punkte auf der Diagonale wurden perfekt vorhergesagt.

In [10]:
echt = np.exp(y_test)
prognose = np.exp(bestes.predict(X_test))
fig = px.scatter(x=echt, y=prognose, labels={"x": "Tatsächliche Miete (€)", "y": "Geschätzte Miete (€)"},
                 title="Geschätzte vs. tatsächliche Miete")
fig.add_shape(type="line", x0=echt.min(), y0=echt.min(), x1=echt.max(), y1=echt.max(), line_color="red")
fig.show()

## 3.11 Deal-Bewertung (Fragen 4 und 5)
Für jede Wohnung berechnen wir die faire Miete und die Abweichung:
Abweichung = (tatsächliche Miete − geschätzte Miete) / geschätzte Miete
Kategorien: Sehr guter Deal (< −15 %), Guter Deal (−15 bis −5 %), Fair (±5 %), Teuer (5 bis 15 %), Zu teuer (> 15 %).

Die Schwellenwerte orientieren sich an der Modellgenauigkeit: Da das Modell im Durchschnitt um ca. 14 % (MAPE) danebenliegt, gelten Abweichungen bis ±10 % als "Fair". Erst ab 10 % (bzw. 25 %) Abweichung sprechen wir von einem (sehr) guten Deal bzw. einer teuren (zu teuren) Wohnung.
Kategorien: Sehr guter Deal (< −25 %), Guter Deal (−25 bis −10 %), Fair (±10 %), Teuer (10 bis 25 %), Zu teuer (> 25 %).

In [11]:
from sklearn.model_selection import cross_val_predict

log_prognose = cross_val_predict(bestes, X, y, cv=5)
df["faire_miete"] = np.exp(log_prognose).round(0)
df["abweichung"] = (df["price"] - df["faire_miete"]) / df["faire_miete"] * 100
df["ersparnis"] = df["faire_miete"] - df["price"]

df["kategorie"] = pd.cut(df["abweichung"],
    bins=[-np.inf, -25, -10, 10, 25, np.inf],
    labels=["Sehr guter Deal", "Guter Deal", "Fair", "Teuer", "Zu teuer"])

df["kategorie"].value_counts()

check(df["faire_miete"].min() > 0, "Alle geschätzten Mieten positiv")
check(df["kategorie"].isna().sum() == 0, "Jedes Inserat hat eine Kategorie")
anteil_fair = (df["kategorie"] == "Fair").mean() * 100
print(f"Anteil fair bepreist: {anteil_fair:.1f} %")

print(df["kategorie"].value_counts())

✅ Alle geschätzten Mieten positiv
✅ Jedes Inserat hat eine Kategorie
Anteil fair bepreist: 44.4 %
kategorie
Fair               16
Zu teuer            7
Guter Deal          6
Teuer               4
Sehr guter Deal     3
Name: count, dtype: int64


## 3.12 Verteilung der Abweichungen
Das Histogramm zeigt, wie viele Inserate unter bzw. über der Modell-Schätzung liegen.

In [12]:
farben = {"Sehr guter Deal": "#1a9850", "Guter Deal": "#91cf60", "Fair": "#bdbdbd",
          "Teuer": "#fc8d59", "Zu teuer": "#d73027"}
reihenfolge = ["Sehr guter Deal", "Guter Deal", "Fair", "Teuer", "Zu teuer"]

fig = px.histogram(df, x="abweichung", nbins=80, color="kategorie",
                   color_discrete_map=farben, category_orders={"kategorie": reihenfolge},
                   labels={"abweichung": "Abweichung von der fairen Miete (%)", "kategorie": "Kategorie"},
                   title="Verteilung der Abweichungen")
fig.update_xaxes(range=[-60, 100])
fig.update_yaxes(title="Anzahl Inserate")
fig.show()

print("Inserate über +100 % (nicht im Bild):", (df["abweichung"] > 100).sum())

Inserate über +100 % (nicht im Bild): 2


## 3.13 Muster nach Bezirk, Anbieter und Plattform (Frage 5)
Wir berechnen pro Gruppe den Anteil der Kategorien in Prozent.

In [13]:
for spalte in ["bezirk", "is_private", "source"]:
    tabelle = pd.crosstab(df[spalte], df["kategorie"], normalize="index") * 100
    display(tabelle.round(1))

kategorie,Sehr guter Deal,Guter Deal,Fair,Teuer,Zu teuer
bezirk,,,,,
2,0.0,0.0,0.0,50.0,50.0
3,0.0,11.1,55.6,22.2,11.1
4,0.0,0.0,100.0,0.0,0.0
5,100.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,100.0
10,0.0,0.0,50.0,0.0,50.0
12,0.0,0.0,100.0,0.0,0.0
13,0.0,0.0,100.0,0.0,0.0
14,0.0,100.0,0.0,0.0,0.0


kategorie,Sehr guter Deal,Guter Deal,Fair,Teuer,Zu teuer
is_private,,,,,
0,6.7,16.7,43.3,10.0,23.3
1,16.7,16.7,50.0,16.7,0.0


kategorie,Sehr guter Deal,Guter Deal,Fair,Teuer,Zu teuer
source,,,,,
immoscout24,0.0,11.1,44.4,11.1,33.3
willhaben,11.1,18.5,44.4,11.1,14.8


## 3.14 Top-Deals
Wir ranken alle Inserate nach der Ersparnis gegenüber der fairen Miete, einmal in Euro und einmal in Prozent. Das Ranking in Euro bevorzugt große Wohnungen (20 % von 4.000 € sind mehr als 20 % von 1.000 €), das Ranking in Prozent zeigt relative Schnäppchen unabhängig von der Größe.

In [14]:
spalten = ["title", "bezirk", "area_m2", "price", "faire_miete", "ersparnis", "abweichung", "kategorie"]

print("Top 10 nach Ersparnis in €:")
display(df.sort_values("ersparnis", ascending=False)[spalten].head(10).round(1))

print("\nTop 10 nach Abweichung in %:")
display(df.sort_values("abweichung")[spalten].head(10).round(1))

Top 10 nach Ersparnis in €:


,title,bezirk,area_m2,price,faire_miete,ersparnis,abweichung,kategorie
9,Margareten | 4-Zimmer-Wohnung mit 67 m² | WG-t...,5,67.0,1390.0,2007.0,617.0,-30.7,Sehr guter Deal
2,vollmöbliert im Herzen Margaretens - ab Jänner...,5,59.0,774.4,1341.0,566.6,-42.3,Sehr guter Deal
32,unbefristete MIETWOHNUNGEN mit KAUFOPTION,14,124.1,2662.3,3090.0,427.7,-13.8,Guter Deal
24,Voll ausgestattete Single Wohnung,16,12.0,650.0,985.0,335.0,-34.0,Sehr guter Deal
12,Wohnen in Ottakring - U3 Kendlerstraße vor der...,16,39.0,849.0,1087.0,238.0,-21.9,Guter Deal
25,Wunderschöne Altbauwohnung in Oberdöblinger Zi...,19,128.0,2500.0,2722.0,222.0,-8.2,Fair
19,VILLAGE IM DRITTEN | BAUFELD 4A | 3 ZIMMER | B...,3,67.0,1499.0,1695.0,196.0,-11.6,Guter Deal
17,VILLAGE IM DRITTEN | BAUFELD 4A | MIETWOHNUNGE...,3,100.0,2379.0,2562.0,183.0,-7.1,Fair
29,wunderschöne Dachgeschoßwohnung,4,104.0,2490.0,2666.0,176.0,-6.6,Fair
28,Sehr schöne Wohnung in Ruhelage mit Grünblick ...,21,48.0,945.0,1111.0,166.0,-14.9,Guter Deal



Top 10 nach Abweichung in %:


,title,bezirk,area_m2,price,faire_miete,ersparnis,abweichung,kategorie
2,vollmöbliert im Herzen Margaretens - ab Jänner...,5,59.0,774.4,1341.0,566.6,-42.3,Sehr guter Deal
24,Voll ausgestattete Single Wohnung,16,12.0,650.0,985.0,335.0,-34.0,Sehr guter Deal
9,Margareten | 4-Zimmer-Wohnung mit 67 m² | WG-t...,5,67.0,1390.0,2007.0,617.0,-30.7,Sehr guter Deal
12,Wohnen in Ottakring - U3 Kendlerstraße vor der...,16,39.0,849.0,1087.0,238.0,-21.9,Guter Deal
28,Sehr schöne Wohnung in Ruhelage mit Grünblick ...,21,48.0,945.0,1111.0,166.0,-14.9,Guter Deal
32,unbefristete MIETWOHNUNGEN mit KAUFOPTION,14,124.1,2662.3,3090.0,427.7,-13.8,Guter Deal
11,Stadlau | 2-Zimmer-Neubauwohnung mit Südbalkon...,22,41.0,961.4,1109.0,147.6,-13.3,Guter Deal
0,"2-Zimmer Mietwohnung in Perfektastraße, beste ...",23,50.0,970.0,1110.0,140.0,-12.6,Guter Deal
19,VILLAGE IM DRITTEN | BAUFELD 4A | 3 ZIMMER | B...,3,67.0,1499.0,1695.0,196.0,-11.6,Guter Deal
25,Wunderschöne Altbauwohnung in Oberdöblinger Zi...,19,128.0,2500.0,2722.0,222.0,-8.2,Fair


## 3.15 Stabilität über die Zeit (Frage 6)
Wir trainieren nur mit älteren Inseraten (die ersten 80 % nach `first_seen`) und testen mit den neuesten 20 %. Ist der Fehler ähnlich wie beim zufälligen Split, bleibt das Modell auch bei neuen Inseraten stabil.

In [15]:
df_zeit = df.sort_values("first_seen")
grenze = int(len(df_zeit) * 0.8)
alt, neu = df_zeit.index[:grenze], df_zeit.index[grenze:]

bestes.fit(X.loc[alt], y.loc[alt])
prog_neu = np.exp(bestes.predict(X.loc[neu]))
mape_neu = mean_absolute_percentage_error(np.exp(y.loc[neu]), prog_neu) * 100
print(f"MAPE bei neuen Inseraten: {mape_neu:.1f} %")

mape_vorher = ergebnisse.loc[ergebnisse["Modell"] == bester_name, "MAPE (%)"].iloc[0]
check(abs(mape_neu - mape_vorher) < 5, f"Fehler bei neuen Inseraten ähnlich wie vorher ({mape_vorher:.1f} % vs. {mape_neu:.1f} %)")

MAPE bei neuen Inseraten: 22.7 %
✅ Fehler bei neuen Inseraten ähnlich wie vorher (18.1 % vs. 22.7 %)


In [16]:
import sys; sys.path.append("..")
from export_to_clickhouse import export_all
export_all(df, ergebnisse, wichtigkeit, effekte, bester_name,
           mape_neu=mape_neu, basis_bezirk=basis)

36 Inserate nach ClickHouse geschrieben
Modellkennzahlen angehängt
Kontrolle: 36 Zeilen in wohnungsmarkt.listings


## 3.16 Ergebnisse zurück in MongoDB schreiben
Laut Konzept werden die Bewertungen (geschätzte Miete, Abweichung, Kategorie) pro Inserat in MongoDB gespeichert, damit das Dashboard sie nutzen kann.

In [17]:
from pymongo import MongoClient

coll = MongoClient("mongodb://localhost:27017/")["wohnungsmarkt"]["listings_clean"]

coll.update_many({}, {"$unset": {"faire_miete": "", "abweichung": "", "kategorie": ""}})

for _, w in df.iterrows():
    coll.update_one({"_id": w["_id"]}, {"$set": {
        "faire_miete": float(w["faire_miete"]),
        "abweichung": float(w["abweichung"]),
        "kategorie": str(w["kategorie"])}})
check(True, f"{len(df)} Bewertungen gespeichert")

gespeichert = coll.count_documents({"kategorie": {"$exists": True}})
check(gespeichert == len(df), f"{gespeichert} Bewertungen in MongoDB gespeichert")

✅ 36 Bewertungen gespeichert
✅ 36 Bewertungen in MongoDB gespeichert


## 3.17 Ergebnisse und Limitationen

**Vorhersagequalität (Frage 2):** Nach dem Entfernen von Sonderfällen (WG-Zimmer, Häuser, Mietkauf u. a.) umfasst der Datensatz 2.338 Inserate. Das beste Modell (Random Forest) erreicht auf dem Testset einen durchschnittlichen Fehler von 13,9 % (MAPE) bzw. __ € (MAE) und ein R² von 0,76. In der 5-fachen Cross-Validation erreichen Random Forest und Gradient Boosting ein R² von __, die lineare Baseline __. Die Baummodelle sind damit klar besser, untereinander aber praktisch gleichwertig. Mieten bis ca. 3.000 € werden zuverlässig geschätzt, sehr teure Wohnungen eher unterschätzt.

**Einflussfaktoren (Frage 3):** Wichtigster Faktor ist mit Abstand die Fläche, gefolgt von Öffi-Score, Bezirk und Freifläche. Zimmeranzahl und Plattform spielen kaum eine Rolle. Laut log-linearem Modell erhöht jeder zusätzliche m² die Miete um ca. __ %, ein zusätzliches Zimmer um __ %, eine Freifläche um __ % und ein Öffi-Score-Punkt um __ %. Gegenüber dem Vergleichsbezirk Donaustadt ist die Innere Stadt ca. __ % teurer, Simmering ca. __ % günstiger. Bezirk und Öffi-Score messen teilweise dasselbe (Lage), weshalb der Bezirk in der Permutation Importance geringer erscheint.

**Schwellenwerte (Frage 4):** Da das Modell im Schnitt um ca. 14 % abweicht, wären enge Schwellen (±5 %) nicht aussagekräftig. Wir legen daher fest: Fair bis ±10 %, Guter Deal bzw. Teuer bei 10 bis 25 % Abweichung, Sehr guter Deal bzw. Zu teuer über 25 %.

**Bewertung der Inserate (Frage 5):** 49 % der Inserate sind fair bepreist, 28 % sind (sehr) gute Deals und 23 % (zu) teuer. Die Verteilung ist rechtsschief: Überteuerte Angebote weichen stärker ab als günstige. Die meisten Deals finden sich in Außenbezirken wie Simmering, Penzing und Favoriten, die meisten überteuerten Angebote in Mariahilf, Wieden und Währing. Private Vermieter bieten etwa dreimal so oft sehr gute Deals an wie gewerbliche. Zwischen den Plattformen gibt es kaum Unterschiede. Das Ergebnis ist robust: Auch vor dem Entfernen der Sonderfälle war die Aufteilung nahezu identisch.

**Stabilität (Frage 6):** Wird das Modell nur mit älteren Inseraten trainiert, liegt der Fehler bei den neuesten 20 % bei __ %, also auf demselben Niveau wie beim zufälligen Split. Das Modell ist im beobachteten Zeitraum stabil.

**Limitationen:**
- Es werden Angebots- statt tatsächlich gezahlter Mieten verwendet.
- Befristung, Zustand, Baujahr und Ausstattung werden nicht erfasst. Der Freiflächen-Effekt enthält vermutlich teilweise einen Neubau-Effekt.
- Der Öffi-Score ist vereinfacht (ohne Gewichtung nach Distanz), Bezirkskennzahlen (Einkommen) fehlen noch.
- Koordinaten bzw. Bezirkszuordnung sind teilweise ungenau, vereinzelt gibt es Duplikate.
- Sonderfälle werden per Schlagwortsuche erkannt, das ist nicht fehlerfrei.
- Private Inserate enthalten eventuell teils Mieten ohne Betriebskosten.
- Der Beobachtungszeitraum ist kurz. Das Modell sollte regelmäßig mit neuen Daten nachtrainiert werden.